In [ ]:
# ============================================================
# CONFIG — change MODE here before running
# ============================================================
MODE = "full"   # "mini" = smoke test (~2 min), "full" = production run

SEED = 42
OUTPUT_BASE = "results/interpretability_v2"
N_QUAL  = {"mini": 3, "full": 15}[MODE]   # qualitative samples per error group
N_QUANT = {"mini": 6, "full": 100}[MODE]   # total quantitative samples (split equally: N_QUANT//2 per error group)
N_PRED  = {"mini": 20, "full": None}[MODE] # examples for prediction pass (None = all)

import os, gc, json, time
import torch
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from transformers import AutoModelForMultipleChoice, AutoTokenizer, Trainer, TrainingArguments
from captum.attr import IntegratedGradients
from datasets import load_dataset
from data_loader import get_dataloaders
from sklearn.metrics.pairwise import cosine_similarity as sk_cosine

os.makedirs(OUTPUT_BASE, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"

# Guard: must be run from project root
print(f"MODE={MODE}  device={device}  N_qual={N_QUAL}  N_quant={N_QUANT}")

MODE=mini  device=cuda  N_qual=3  N_quant=6


In [5]:
# ============================================================
# MODEL REGISTRY
# ============================================================
MODELS = [
    {
        "name": "vanilla_L6_A0p7_T20",
        "path": "results/training_runs/vanilla_kd_grid_search/vanilla_L6_A0p7_T20/checkpoint-3975",
        "strategy": "vanilla_kd", "layers": 6
    },
    {
        "name": "pkd_skip_L6_B1000",
        "path": "results/training_runs/pkd_skip_grid_search/pkd_skip_L6_B1000/checkpoint-3975",
        "strategy": "pkd_skip", "layers": 6
    },
    {
        "name": "pkd_skip_L4_B100",
        "path": "results/training_runs/pkd_skip_grid_search/pkd_skip_L4_B100/checkpoint-3975",
        "strategy": "pkd_skip", "layers": 4
    },
    {
        "name": "pkd_skip_L3_B1000",
        "path": "results/training_runs/pkd_skip_grid_search/pkd_skip_L3_B1000/checkpoint-5300",
        "strategy": "pkd_skip", "layers": 3
    },
    {
        "name": "pkd_skip_L2_B500",
        "path": "results/training_runs/pkd_skip_grid_search/pkd_skip_L2_B500/checkpoint-5300",
        "strategy": "pkd_skip", "layers": 2
    },
    {
        "name": "pkd_last_L6_B500",
        "path": "results/training_runs/pkd_last_grid_search/pkd_last_L6_B500/checkpoint-3975",
        "strategy": "pkd_last", "layers": 6
    },
    {
        "name": "pkd_last_L4_B500",
        "path": "results/training_runs/pkd_last_grid_search/pkd_last_L4_B500/checkpoint-3975",
        "strategy": "pkd_last", "layers": 4
    },
    {
        "name": "pkd_last_L3_B500",
        "path": "results/training_runs/pkd_last_grid_search/pkd_last_L3_B500/checkpoint-3975",
        "strategy": "pkd_last", "layers": 3
    },
    {
        "name": "pkd_last_L2_B500",
        "path": "results/training_runs/pkd_last_grid_search/pkd_last_L2_B500/checkpoint-5300",
        "strategy": "pkd_last", "layers": 2
    },
]

TEACHER_PATH = "results/training_runs/fine_tuned_base_bert_legal_teacher/run_lr_1e-05/checkpoint-1325"

# Verify all paths exist
print("Verifying model paths...")
all_ok = True
for m in [{"name": "teacher", "path": TEACHER_PATH}] + MODELS:
    exists = Path(m["path"]).exists()
    status = "OK" if exists else "MISSING"
    print(f"  [{status}] {m['name']}")
    if not exists:
        all_ok = False
print("All paths OK" if all_ok else "WARNING: Some paths are missing!")

Verifying model paths...
  [OK] teacher
  [OK] vanilla_L6_A0p7_T20
  [OK] pkd_skip_L6_B1000
  [OK] pkd_skip_L4_B100
  [OK] pkd_skip_L3_B1000
  [OK] pkd_skip_L2_B500
  [OK] pkd_last_L6_B500
  [OK] pkd_last_L4_B500
  [OK] pkd_last_L3_B500
  [OK] pkd_last_L2_B500
All paths OK


In [6]:
# ============================================================
# DATA LOADING
# ============================================================
tokenizer = AutoTokenizer.from_pretrained("nlpaueb/legal-bert-base-uncased")

# Tokenized test set (for model inference & attribution)
datasets_all = get_dataloaders(tokenizer, return_dict=True)
test_dataset_tokenized = datasets_all['test']

# Raw test set (for metadata: case text, holdings)
test_dataset_raw = load_dataset(
    'csv',
    data_files={'test': 'data/casehold/test.csv'},
    split='test'
)

# In mini mode, slice to first N_PRED examples
if N_PRED is not None:
    test_dataset_tokenized = test_dataset_tokenized.select(range(N_PRED))
    test_dataset_raw = test_dataset_raw.select(range(N_PRED))

print(f"Test set size: {len(test_dataset_tokenized)} examples")
print(f"Raw test set size: {len(test_dataset_raw)} examples")

# Sanity check alignment
# Order is guaranteed: get_dataloaders uses datasets.map() with no shuffle on test split.
# Both datasets read from the same test.csv in identical row order.
assert len(test_dataset_tokenized) == len(test_dataset_raw), "Sizes must match!"
print("Alignment check passed.")

Generating test split: 0 examples [00:00, ? examples/s]

Test set size: 20 examples
Raw test set size: 20 examples
Alignment check passed.


In [7]:
# ============================================================
# LOAD TEACHER (stays in memory the whole notebook)
# ============================================================
print("Loading teacher model...")
teacher_model = AutoModelForMultipleChoice.from_pretrained(TEACHER_PATH).to(device).eval()
print(f"Teacher loaded: {len(teacher_model.bert.encoder.layer)} layers")

Loading teacher model...
Teacher loaded: 12 layers


In [ ]:
# ============================================================
# HELPER FUNCTIONS (unchanged from v1)
# ============================================================

def get_attributions(model, input_ids, attention_mask, method='ig'):
    """Get token attributions for a single input. Returns (seq_len,) numpy array."""
    input_ids = input_ids.unsqueeze(0).to(device)
    attention_mask = attention_mask.unsqueeze(0).to(device)

    def forward_wrapper(input_embeds):
        input_embeds = input_embeds.unsqueeze(1)
        attention_mask_expanded = attention_mask.unsqueeze(1)
        outputs = model(inputs_embeds=input_embeds, attention_mask=attention_mask_expanded)
        return outputs.logits.squeeze(1)

    with torch.no_grad():
        embeddings = model.bert.embeddings(input_ids.squeeze(1))
    embeddings = embeddings.detach().clone().unsqueeze(0)
    embeddings.requires_grad = True

    attr_method = IntegratedGradients(forward_wrapper)
    attributions = attr_method.attribute(embeddings, n_steps=50)
    return attributions.sum(dim=-1).squeeze().detach().cpu().numpy()


def filter_tokens_and_attributions(tokens, attributions):
    """Filter out [CLS]/[SEP]/[PAD]/subwords/punctuation."""
    filtered_tokens, filtered_attr, filtered_indices = [], [], []
    for idx, (token, attr) in enumerate(zip(tokens, attributions)):
        if token in ['[CLS]', '[SEP]', '[PAD]', '[UNK]']:
            continue
        if token.startswith('##'):
            continue
        if len(token) == 1 and not token.isdigit():
            continue
        if all(c in '.,;:!?()[]{}"\'-/' for c in token):
            continue
        filtered_tokens.append(token)
        filtered_attr.append(attr)
        filtered_indices.append(idx)
    return filtered_tokens, np.array(filtered_attr), filtered_indices


def save_attribution_heatmap(tokens, teacher_attr, student_attr, output_path,
                              max_tokens=30, title="Attribution Comparison"):
    """Save side-by-side teacher/student attribution heatmap as PNG."""
    tokens = tokens[:max_tokens]
    teacher_attr = teacher_attr[:max_tokens]
    student_attr = student_attr[:max_tokens]

    teacher_norm = teacher_attr / (np.abs(teacher_attr).max() + 1e-10)
    student_norm = student_attr / (np.abs(student_attr).max() + 1e-10)

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 8), dpi=150)
    sns.heatmap(teacher_norm.reshape(1, -1), cmap='RdBu_r', center=0, vmin=-1, vmax=1,
                xticklabels=tokens, yticklabels=['Teacher'],
                cbar_kws={'label': 'Attribution'}, ax=ax1)
    ax1.set_title('Teacher Attributions', fontsize=14)
    ax1.set_xticklabels(tokens, rotation=90, fontsize=8)
    sns.heatmap(student_norm.reshape(1, -1), cmap='RdBu_r', center=0, vmin=-1, vmax=1,
                xticklabels=tokens, yticklabels=['Student'],
                cbar_kws={'label': 'Attribution'}, ax=ax2)
    ax2.set_title('Student Attributions', fontsize=14)
    ax2.set_xticklabels(tokens, rotation=90, fontsize=8)
    plt.suptitle(title, fontsize=14, y=1.02)
    plt.tight_layout()
    plt.savefig(output_path, bbox_inches='tight', dpi=150)
    plt.close(fig)


def save_top15_bar_chart(tokens, teacher_attr, student_attr, output_path, top_k=15):
    """Save top-k combined tokens bar chart as PNG."""
    teacher_top_idx = np.argsort(np.abs(teacher_attr))[-top_k:][::-1]
    student_top_idx = np.argsort(np.abs(student_attr))[-top_k:][::-1]
    all_top_idx = sorted(set(teacher_top_idx) | set(student_top_idx))

    fig, ax = plt.subplots(figsize=(12, 6), dpi=150)
    x = np.arange(len(all_top_idx))
    width = 0.35
    ax.bar(x - width/2, [teacher_attr[i] for i in all_top_idx], width, label='Teacher', alpha=0.8)
    ax.bar(x + width/2, [student_attr[i] for i in all_top_idx], width, label='Student', alpha=0.8)
    ax.set_ylabel('Attribution Score', fontsize=10)
    ax.set_title(f'Top-{top_k} Most Important Tokens (Combined)', fontsize=14)
    ax.set_xticks(x)
    ax.set_xticklabels([tokens[i] for i in all_top_idx], rotation=45, ha='right', fontsize=8)
    ax.legend()
    ax.axhline(y=0, color='k', linestyle='-', linewidth=0.5)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(output_path, bbox_inches='tight', dpi=150)
    plt.close(fig)


def extract_example_metadata(raw_example, label_idx):
    """Extract context + holdings from raw CSV row."""
    return {
        'context': raw_example['1'],
        'holdings': [raw_example[str(i+2)] for i in range(5)],
        'label': int(float(raw_example['12']))
    }


def compute_cosine_similarity(vec1, vec2):
    return sk_cosine(vec1.reshape(1, -1), vec2.reshape(1, -1))[0, 0]


def attribution_entropy(attr):
    """Shannon entropy of the absolute attribution distribution.

    Measures how focused vs. diffuse the model's attention is.
    High entropy = spread across many tokens (diffuse).
    Low entropy  = concentrated on a few tokens (focused).
    Returns a float in [0, log(N)] where N = len(attr).
    """
    abs_attr = np.abs(attr)
    total = abs_attr.sum()
    if total == 0:
        return 0.0
    p = abs_attr / total
    p = p[p > 0]
    return float(-np.sum(p * np.log(p)))


print("v1 helper functions loaded (+ attribution_entropy).")

In [ ]:
# ============================================================
# NEW: PREDICTION PASS WITH CONFIDENCE + TIMING
# ============================================================

def run_prediction_pass(model, model_name, dataset_tokenized, dataset_raw):
    """
    Run full forward pass over dataset. Writes predictions.csv.
    Skips if file already exists and has prob columns (crash recovery).
    Re-runs if prob_0..prob_4 columns are missing (upgrade detection).

    Returns: pd.DataFrame with columns:
      example_idx, true_label, predicted_label, is_correct,
      confidence, inference_time_sec, prob_0..prob_4
    """
    out_dir = Path(OUTPUT_BASE) / model_name
    out_dir.mkdir(parents=True, exist_ok=True)
    csv_path = out_dir / "predictions.csv"

    if csv_path.exists():
        existing = pd.read_csv(csv_path)
        if 'prob_0' in existing.columns:
            print(f"  [SKIP] predictions.csv already exists for {model_name}")
            return existing
        else:
            print(f"  [UPGRADE] predictions.csv missing prob columns — re-running for {model_name}")

    print(f"  Running prediction pass for {model_name} ({len(dataset_tokenized)} examples)...")

    eval_args = TrainingArguments(
        output_dir="./temp_eval_v2",
        per_device_eval_batch_size=4,
        dataloader_num_workers=0,
        fp16=(device == "cuda"),
        no_cuda=(device != "cuda"),
        remove_unused_columns=False,
        report_to="none",
    )
    trainer = Trainer(model=model, args=eval_args)

    start = time.time()
    pred_output = trainer.predict(dataset_tokenized)
    elapsed = time.time() - start

    logits = pred_output.predictions          # (N, 5)
    true_labels = pred_output.label_ids       # (N,)
    predicted_labels = np.argmax(logits, axis=1)

    # Softmax for confidence and full probability distribution
    exp_logits = np.exp(logits - logits.max(axis=1, keepdims=True))
    probs = exp_logits / exp_logits.sum(axis=1, keepdims=True)  # (N, 5)
    confidence = probs.max(axis=1)

    per_example_time = elapsed / len(dataset_tokenized)

    rows = []
    for i in range(len(dataset_tokenized)):
        rows.append({
            'example_idx': i,
            'true_label': int(true_labels[i]),
            'predicted_label': int(predicted_labels[i]),
            'is_correct': bool(true_labels[i] == predicted_labels[i]),
            'confidence': float(confidence[i]),
            'inference_time_sec': per_example_time,
            'prob_0': float(probs[i, 0]),
            'prob_1': float(probs[i, 1]),
            'prob_2': float(probs[i, 2]),
            'prob_3': float(probs[i, 3]),
            'prob_4': float(probs[i, 4]),
        })

    df = pd.DataFrame(rows)
    df.to_csv(csv_path, index=False)

    acc = df['is_correct'].mean()
    print(f"  Accuracy: {acc:.4f} | Avg confidence: {df['confidence'].mean():.4f} | Time/example: {per_example_time*1000:.1f}ms")
    return df


print("Prediction pass helper defined.")

In [10]:
# Smoke test: run prediction pass on teacher with mini data
print("=== SMOKE TEST: teacher prediction pass ===")
teacher_preds = run_prediction_pass(
    teacher_model, "_SMOKE_TEST_teacher",
    test_dataset_tokenized, test_dataset_raw
)
print(teacher_preds.head(3))
assert 'is_correct' in teacher_preds.columns
assert 'confidence' in teacher_preds.columns
assert len(teacher_preds) == len(test_dataset_tokenized)
print("Smoke test PASSED.")

=== SMOKE TEST: teacher prediction pass ===
  Running prediction pass for _SMOKE_TEST_teacher (20 examples)...


  Accuracy: 0.7000 | Avg confidence: 0.8036 | Time/example: 31.5ms
   example_idx  true_label  predicted_label  is_correct  confidence  \
0            0           1                1        True    0.760209   
1            1           2                2        True    0.993169   
2            2           0                0        True    0.859679   

   inference_time_sec  
0            0.031547  
1            0.031547  
2            0.031547  
Smoke test PASSED.


In [11]:
# ============================================================
# NEW: ERROR-GROUP SAMPLING
# ============================================================

def sample_by_error_group(predictions_df, n, seed=SEED):
    """
    Sample n indices from each error group.

    Returns:
        correct_indices: list of example_idx values
        incorrect_indices: list of example_idx values

    If fewer than n examples exist in a group, returns all available.
    """
    np.random.seed(seed)

    correct_pool = predictions_df[predictions_df['is_correct'] == True]['example_idx'].values
    incorrect_pool = predictions_df[predictions_df['is_correct'] == False]['example_idx'].values

    n_correct = min(n, len(correct_pool))
    n_incorrect = min(n, len(incorrect_pool))

    correct_sample = np.random.choice(correct_pool, n_correct, replace=False).tolist()
    incorrect_sample = np.random.choice(incorrect_pool, n_incorrect, replace=False).tolist()

    print(f"  Sampled {n_correct} correct, {n_incorrect} incorrect examples")
    return correct_sample, incorrect_sample


print("Error-group sampling helper defined.")

# Quick unit test
_fake_preds = pd.DataFrame({
    'example_idx': range(10),
    'is_correct': [True]*7 + [False]*3
})
_c, _i = sample_by_error_group(_fake_preds, n=5, seed=42)
assert len(_c) == 5
assert len(_i) == 3   # only 3 incorrect available
print("Unit test PASSED.")

Error-group sampling helper defined.
  Sampled 5 correct, 3 incorrect examples
Unit test PASSED.


In [ ]:
# ============================================================
# NEW: ATTRIBUTION ANALYSIS WITH ERROR-GROUP LABELING
# ============================================================

def run_qualitative_for_group(teacher_model, student_model, student_name,
                               example_indices, error_group,
                               dataset_tokenized, dataset_raw,
                               predictions_df):
    """
    Run qualitative attribution analysis for one error group (correct/incorrect).

    For CORRECT predictions: both teacher and student attribute on the true-label input.
    For INCORRECT predictions: teacher attributes on true-label input; student attributes
    on its predicted (wrong) choice. This reveals what the student was "looking at" when
    it went wrong, compared to what the teacher focuses on for the correct answer.

    Saves heatmaps + metadata to:
      results/interpretability_v2/{student_name}/qualitative/{error_group}/sample_NNN/
    """
    group_dir = Path(OUTPUT_BASE) / student_name / "qualitative" / error_group
    group_dir.mkdir(parents=True, exist_ok=True)

    for sample_num, idx in enumerate(example_indices):
        sample_dir = group_dir / f"sample_{sample_num:03d}"
        sample_dir.mkdir(exist_ok=True)

        example_tok = dataset_tokenized[idx]
        example_raw = dataset_raw[idx]
        pred_row = predictions_df[predictions_df['example_idx'] == idx].iloc[0]

        true_label = int(example_tok['labels'].item())
        predicted_label = int(pred_row['predicted_label'])
        is_correct = bool(pred_row['is_correct'])

        # Teacher always attributes on the correct-answer input
        teacher_choice = true_label
        # Student attributes on what IT chose (wrong choice when incorrect)
        student_choice = true_label if is_correct else predicted_label

        tokens = tokenizer.convert_ids_to_tokens(example_tok['input_ids'][teacher_choice])

        teacher_attr = get_attributions(teacher_model, example_tok['input_ids'][teacher_choice],
                                        example_tok['attention_mask'][teacher_choice], 'ig')
        student_attr = get_attributions(student_model, example_tok['input_ids'][student_choice],
                                        example_tok['attention_mask'][student_choice], 'ig')

        filtered_tokens, filtered_teacher, _ = filter_tokens_and_attributions(tokens, teacher_attr)
        _, filtered_student, _ = filter_tokens_and_attributions(tokens, student_attr)

        metadata = extract_example_metadata(example_raw, true_label)
        metadata.update({
            'example_idx': int(idx),
            'sample_num': sample_num,
            'error_group': error_group,
            'is_correct': is_correct,
            'confidence': float(pred_row['confidence']),
            'inference_time_sec': float(pred_row['inference_time_sec']),
            'teacher_attribution_choice': teacher_choice,
            'student_attribution_choice': student_choice,
        })

        with open(sample_dir / 'metadata.json', 'w', encoding='utf-8') as f:
            json.dump(metadata, f, indent=2, ensure_ascii=False)

        save_attribution_heatmap(
            filtered_tokens, filtered_teacher, filtered_student,
            sample_dir / 'attribution_heatmap_filtered.png',
            max_tokens=30,
            title=f"{student_name} [{error_group}] Sample {sample_num}"
        )
        save_top15_bar_chart(
            filtered_tokens, filtered_teacher, filtered_student,
            sample_dir / 'top15_tokens_combined.png',
            top_k=15
        )

    print(f"    Qualitative [{error_group}]: {len(example_indices)} samples saved to {group_dir}")


def run_quantitative_for_group(teacher_model, student_model, student_name,
                                example_indices, error_group, dataset_tokenized,
                                predictions_df):
    """
    Run quantitative attribution metrics for one error group.

    For CORRECT predictions: both models attribute on the true-label input.
    For INCORRECT predictions: teacher uses true_label; student uses predicted_label.

    Metrics: cosine similarity, Pearson correlation, top-10 token overlap,
             attribution entropy (teacher and student separately).

    Saves to:
      results/interpretability_v2/{student_name}/quantitative_{error_group}.csv
      results/interpretability_v2/{student_name}/summary_stats_{error_group}.txt
    """
    results = []
    for idx in example_indices:
        example = dataset_tokenized[idx]
        true_label = int(example['labels'].item())
        pred_row = predictions_df[predictions_df['example_idx'] == idx].iloc[0]
        is_correct = bool(pred_row['is_correct'])
        student_label = true_label if is_correct else int(pred_row['predicted_label'])

        teacher_attr = get_attributions(teacher_model, example['input_ids'][true_label],
                                        example['attention_mask'][true_label], 'ig')
        student_attr = get_attributions(student_model, example['input_ids'][student_label],
                                        example['attention_mask'][student_label], 'ig')

        cosine_sim = compute_cosine_similarity(teacher_attr, student_attr)
        corr = np.corrcoef(teacher_attr, student_attr)[0, 1]
        k = 10
        teacher_top = set(np.argsort(np.abs(teacher_attr))[-k:])
        student_top = set(np.argsort(np.abs(student_attr))[-k:])
        overlap = len(teacher_top & student_top) / k

        results.append({
            'example_idx': int(idx),
            'error_group': error_group,
            'cosine_similarity': cosine_sim,
            'correlation': corr,
            'top10_overlap': overlap,
            'teacher_entropy': attribution_entropy(teacher_attr),
            'student_entropy': attribution_entropy(student_attr),
        })

    df = pd.DataFrame(results)
    out_dir = Path(OUTPUT_BASE) / student_name
    csv_path = out_dir / f"quantitative_{error_group}.csv"
    df.to_csv(csv_path, index=False)

    summary_path = out_dir / f"summary_stats_{error_group}.txt"
    with open(summary_path, 'w') as f:
        f.write(f"Quantitative Summary: {student_name} [{error_group}]\n{'='*50}\n\n")
        for col in ['cosine_similarity', 'correlation', 'top10_overlap',
                    'teacher_entropy', 'student_entropy']:
            f.write(f"{col}:\n  Mean: {df[col].mean():.4f}\n  Std:  {df[col].std():.4f}\n\n")

    print(f"    Quantitative [{error_group}]: {len(results)} samples | "
          f"cosine={df['cosine_similarity'].mean():.3f} | "
          f"corr={df['correlation'].mean():.3f} | "
          f"overlap={df['top10_overlap'].mean():.3f} | "
          f"H_t={df['teacher_entropy'].mean():.3f} | "
          f"H_s={df['student_entropy'].mean():.3f}")
    return df


print("Attribution analysis helpers defined.")

In [ ]:
# ============================================================
# MINI END-TO-END TEST (verifies all helpers work together)
# ============================================================
print("=== MINI END-TO-END TEST ===")
_test_model_cfg = MODELS[0]   # vanilla_L6
_test_student = AutoModelForMultipleChoice.from_pretrained(_test_model_cfg["path"]).to(device).eval()

_preds = run_prediction_pass(_test_student, f"_TEST_{_test_model_cfg['name']}",
                              test_dataset_tokenized, test_dataset_raw)

_correct_idx, _incorrect_idx = sample_by_error_group(_preds, n=N_QUAL)

print("[1/2] Qualitative correct...")
run_qualitative_for_group(teacher_model, _test_student, f"_TEST_{_test_model_cfg['name']}",
                          _correct_idx, "correct",
                          test_dataset_tokenized, test_dataset_raw, _preds)

print("[2/2] Qualitative incorrect...")
if _incorrect_idx:
    run_qualitative_for_group(teacher_model, _test_student, f"_TEST_{_test_model_cfg['name']}",
                              _incorrect_idx, "incorrect",
                              test_dataset_tokenized, test_dataset_raw, _preds)

print("[3/3] Quantitative...")
run_quantitative_for_group(teacher_model, _test_student, f"_TEST_{_test_model_cfg['name']}",
                           _correct_idx + _incorrect_idx, "all",
                           test_dataset_tokenized, _preds)

# Verify outputs exist
_test_dir = Path(OUTPUT_BASE) / f"_TEST_{_test_model_cfg['name']}"
assert (_test_dir / "predictions.csv").exists(), "predictions.csv missing"
assert 'prob_0' in pd.read_csv(_test_dir / "predictions.csv").columns, "prob columns missing"
assert (_test_dir / "qualitative" / "correct" / "sample_000" / "metadata.json").exists()
assert (_test_dir / "qualitative" / "correct" / "sample_000" / "attribution_heatmap_filtered.png").exists()
assert (_test_dir / "qualitative" / "correct" / "sample_000" / "top15_tokens_combined.png").exists()

_quant_df = pd.read_csv(_test_dir / "quantitative_all.csv")
assert 'teacher_entropy' in _quant_df.columns, "teacher_entropy missing"
assert 'student_entropy' in _quant_df.columns, "student_entropy missing"

del _test_student
torch.cuda.empty_cache()

print("=== MINI END-TO-END TEST PASSED ===")

In [ ]:
# ============================================================
# MAIN LOOP: All student models
# ============================================================
print(f"\nStarting main loop over {len(MODELS)} models...")

for model_cfg in MODELS:
    name = model_cfg["name"]
    print(f"\n{'='*60}")
    print(f"Processing: {name} ({model_cfg['layers']} layers, {model_cfg['strategy']})")
    print(f"{'='*60}")

    # Load student
    student = AutoModelForMultipleChoice.from_pretrained(model_cfg["path"]).to(device).eval()
    print(f"  Loaded: {len(student.bert.encoder.layer)} layers")

    # 1. Prediction pass (crash-recoverable; re-runs if prob columns missing)
    print("  [1/3] Prediction pass...")
    preds_df = run_prediction_pass(student, name, test_dataset_tokenized, test_dataset_raw)

    # 2. Sample error groups
    correct_idx, incorrect_idx = sample_by_error_group(preds_df, n=N_QUAL)

    # 3. Qualitative analysis
    print("  [2/3] Qualitative attribution...")
    run_qualitative_for_group(teacher_model, student, name, correct_idx, "correct",
                              test_dataset_tokenized, test_dataset_raw, preds_df)
    if incorrect_idx:
        run_qualitative_for_group(teacher_model, student, name, incorrect_idx, "incorrect",
                                  test_dataset_tokenized, test_dataset_raw, preds_df)
    else:
        print("    WARNING: No incorrect examples — model may be perfect on mini slice")

    # 4. Quantitative analysis (separate per group)
    print("  [3/3] Quantitative attribution...")
    quant_sample_size = N_QUANT // 2
    q_correct, q_incorrect = sample_by_error_group(preds_df, n=quant_sample_size, seed=SEED + 1)
    if q_correct:
        run_quantitative_for_group(teacher_model, student, name, q_correct,
                                   "correct", test_dataset_tokenized, preds_df)
    if q_incorrect:
        run_quantitative_for_group(teacher_model, student, name, q_incorrect,
                                   "incorrect", test_dataset_tokenized, preds_df)

    # Free memory
    del student
    torch.cuda.empty_cache()
    gc.collect()
    print(f"  Done: {name}")

print("\n" + "="*60)
print("MAIN LOOP COMPLETE")
print("="*60)

In [ ]:
# ============================================================
# CROSS-MODEL AGGREGATION
# ============================================================
print("Aggregating results across all models...")

# Load teacher prediction pass (run it now if not done)
print("Running teacher prediction pass...")
teacher_preds = run_prediction_pass(teacher_model, "teacher",
                                    test_dataset_tokenized, test_dataset_raw)

PROB_COLS = ['prob_0', 'prob_1', 'prob_2', 'prob_3', 'prob_4']


def compute_mean_kl_divergence(teacher_df, student_df):
    """Compute mean KL(teacher || student) over all shared examples.

    Uses full softmax distributions (prob_0..prob_4). Returns the average
    divergence in nats. Higher = student distribution is less like teacher's.
    """
    merged = teacher_df[['example_idx'] + PROB_COLS].merge(
        student_df[['example_idx'] + PROB_COLS],
        on='example_idx', suffixes=('_t', '_s')
    )
    t_probs = merged[[c + '_t' for c in PROB_COLS]].values + 1e-10
    s_probs = merged[[c + '_s' for c in PROB_COLS]].values + 1e-10
    kl_per_example = (t_probs * np.log(t_probs / s_probs)).sum(axis=1)
    return float(kl_per_example.mean())


summary_rows = []

# Teacher row (no student comparison — just accuracy + confidence)
teacher_acc = teacher_preds['is_correct'].mean()
teacher_time = teacher_preds['inference_time_sec'].mean()
for group in ['correct', 'incorrect']:
    group_df = teacher_preds[teacher_preds['is_correct'] == (group == 'correct')]
    summary_rows.append({
        'model_name': 'teacher',
        'strategy': 'teacher',
        'layers': 12,
        'error_group': group,
        'n_samples': len(group_df),
        'mean_cosine_sim': None,
        'mean_correlation': None,
        'mean_top10_overlap': None,
        'mean_teacher_entropy': None,
        'mean_student_entropy': None,
        'mean_kl_divergence': None,
        'mean_confidence': group_df['confidence'].mean(),
        'mean_inference_time_sec': teacher_time,
        'overall_accuracy': teacher_acc,
    })

# Student models
for model_cfg in MODELS:
    name = model_cfg["name"]
    out_dir = Path(OUTPUT_BASE) / name
    preds_path = out_dir / "predictions.csv"

    if not preds_path.exists():
        print(f"  WARNING: predictions.csv missing for {name} — skipping")
        continue

    preds_df = pd.read_csv(preds_path)
    overall_acc = preds_df['is_correct'].mean()
    per_example_time = preds_df['inference_time_sec'].mean()

    # KL divergence over full test set (needs prob columns)
    has_probs = (all(c in preds_df.columns for c in PROB_COLS) and
                 all(c in teacher_preds.columns for c in PROB_COLS))
    kl_div = compute_mean_kl_divergence(teacher_preds, preds_df) if has_probs else None

    for group in ['correct', 'incorrect']:
        quant_path = out_dir / f"quantitative_{group}.csv"
        group_preds = preds_df[preds_df['is_correct'] == (group == 'correct')]
        quant_df = pd.read_csv(quant_path) if quant_path.exists() else pd.DataFrame()

        summary_rows.append({
            'model_name': name,
            'strategy': model_cfg['strategy'],
            'layers': model_cfg['layers'],
            'error_group': group,
            'n_samples': len(group_preds),
            'mean_cosine_sim': quant_df['cosine_similarity'].mean() if len(quant_df) else None,
            'mean_correlation': quant_df['correlation'].mean() if len(quant_df) else None,
            'mean_top10_overlap': quant_df['top10_overlap'].mean() if len(quant_df) else None,
            'mean_teacher_entropy': quant_df['teacher_entropy'].mean()
                                    if ('teacher_entropy' in quant_df.columns and len(quant_df)) else None,
            'mean_student_entropy': quant_df['student_entropy'].mean()
                                    if ('student_entropy' in quant_df.columns and len(quant_df)) else None,
            'mean_kl_divergence': kl_div,
            'mean_confidence': group_preds['confidence'].mean() if len(group_preds) else None,
            'mean_inference_time_sec': per_example_time,
            'overall_accuracy': overall_acc,
        })

summary_df = pd.DataFrame(summary_rows)
summary_path = Path(OUTPUT_BASE) / "summary_all_models.csv"
summary_df.to_csv(summary_path, index=False)
print(f"Summary saved: {summary_path}")
print(summary_df[['model_name', 'layers', 'error_group', 'overall_accuracy',
                  'mean_confidence', 'mean_kl_divergence']].to_string())

In [16]:
# ============================================================
# SUMMARY PLOTS
# ============================================================
plots_dir = Path(OUTPUT_BASE) / "plots"
plots_dir.mkdir(exist_ok=True)

# Use only one row per model for accuracy/timing plots
per_model = summary_df.drop_duplicates(subset='model_name').copy()
per_model = per_model.sort_values(['layers', 'strategy'])

strategy_colors = {'teacher': '#2c7bb6', 'vanilla_kd': '#abd9e9',
                   'pkd_skip': '#fdae61', 'pkd_last': '#d7191c'}

# Plot 1: Accuracy by model
fig, ax = plt.subplots(figsize=(12, 5), dpi=150)
colors = [strategy_colors.get(s, 'gray') for s in per_model['strategy']]
bars = ax.bar(per_model['model_name'], per_model['overall_accuracy'], color=colors)
ax.set_xticklabels(per_model['model_name'], rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Test Accuracy')
ax.set_title('Test Accuracy by Model')
ax.set_ylim(0.6, 0.8)
ax.axhline(y=per_model[per_model['strategy']=='teacher']['overall_accuracy'].values[0],
           color='gray', linestyle='--', linewidth=1, label='Teacher baseline')
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=s) for s, c in strategy_colors.items()]
ax.legend(handles=legend_elements, fontsize=8)
plt.tight_layout()
plt.savefig(plots_dir / 'accuracy_by_model.png', bbox_inches='tight', dpi=150)
plt.close(fig)
print("Saved: accuracy_by_model.png")

# Plot 2: Confidence correct vs incorrect
conf_data = summary_df[summary_df['strategy'] != 'teacher'].copy()
conf_pivot = conf_data.pivot(index='model_name', columns='error_group', values='mean_confidence')
fig, ax = plt.subplots(figsize=(12, 5), dpi=150)
x = np.arange(len(conf_pivot))
width = 0.35
ax.bar(x - width/2, conf_pivot.get('correct', [0]*len(conf_pivot)), width,
       label='Correct predictions', color='#2ca02c', alpha=0.8)
ax.bar(x + width/2, conf_pivot.get('incorrect', [0]*len(conf_pivot)), width,
       label='Incorrect predictions', color='#d62728', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(conf_pivot.index, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Mean Max Softmax Probability')
ax.set_title('Model Confidence: Correct vs Incorrect Predictions')
ax.legend()
plt.tight_layout()
plt.savefig(plots_dir / 'confidence_correct_vs_incorrect.png', bbox_inches='tight', dpi=150)
plt.close(fig)
print("Saved: confidence_correct_vs_incorrect.png")

# Plot 3: Cosine similarity by layers (skip vs last)
cosine_data = summary_df[
    (summary_df['strategy'].isin(['pkd_skip', 'pkd_last'])) &
    (summary_df['error_group'] == 'correct') &
    (summary_df['mean_cosine_sim'].notna())
].copy()
fig, ax = plt.subplots(figsize=(8, 5), dpi=150)
for strategy, grp in cosine_data.groupby('strategy'):
    grp_sorted = grp.sort_values('layers')
    ax.plot(grp_sorted['layers'], grp_sorted['mean_cosine_sim'],
            marker='o', label=strategy, linewidth=2)
ax.set_xlabel('Student Layers')
ax.set_ylabel('Mean Cosine Similarity (Teacher vs Student)')
ax.set_title('Attribution Similarity by Layer Count\n(correct predictions)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(plots_dir / 'cosine_sim_by_layers.png', bbox_inches='tight', dpi=150)
plt.close(fig)
print("Saved: cosine_sim_by_layers.png")

# Plot 4: Inference time by model
fig, ax = plt.subplots(figsize=(12, 5), dpi=150)
per_model_sorted = per_model.sort_values('layers')
colors4 = [strategy_colors.get(s, 'gray') for s in per_model_sorted['strategy']]
ax.bar(per_model_sorted['model_name'],
       per_model_sorted['mean_inference_time_sec'] * 1000,   # convert to ms
       color=colors4)
ax.set_xticklabels(per_model_sorted['model_name'], rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Per-Example Inference Time (ms)')
ax.set_title('Inference Time by Model')
legend_elements2 = [Patch(facecolor=c, label=s) for s, c in strategy_colors.items()]
ax.legend(handles=legend_elements2, fontsize=8)
plt.tight_layout()
plt.savefig(plots_dir / 'inference_time_by_model.png', bbox_inches='tight', dpi=150)
plt.close(fig)
print("Saved: inference_time_by_model.png")

print("\nAll 4 plots saved to:", plots_dir)

C:\Users\rhrou\AppData\Local\Temp\ipykernel_87288\2086734518.py:18: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(per_model['model_name'], rotation=45, ha='right', fontsize=8)


Saved: accuracy_by_model.png
Saved: confidence_correct_vs_incorrect.png
Saved: cosine_sim_by_layers.png
Saved: inference_time_by_model.png

All 4 plots saved to: results\interpretability_v2\plots


C:\Users\rhrou\AppData\Local\Temp\ipykernel_87288\2086734518.py:80: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(per_model_sorted['model_name'], rotation=45, ha='right', fontsize=8)


In [17]:
# ============================================================
# PKD-LAST SIMILARITY REPORT (matches format of v1 summary_readable.txt)
# ============================================================
report_path = Path(OUTPUT_BASE) / "pkd_last_similarity_summary.txt"

pkd_last_models = [m for m in MODELS if m['strategy'] == 'pkd_last']

with open(report_path, 'w') as f:
    f.write("Cross-Model Attribution Comparison Summary — PKD-Last\n")
    f.write("="*60 + "\n\n")

    for model_cfg in pkd_last_models:
        name = model_cfg["name"]
        layers = model_cfg["layers"]

        for group in ['correct', 'incorrect']:
            quant_path = Path(OUTPUT_BASE) / name / f"quantitative_{group}.csv"
            if not quant_path.exists():
                continue
            df = pd.read_csv(quant_path)

            f.write(f"{name} ({layers} layers) — [{group}]\n")
            f.write("-" * 40 + "\n")
            f.write(f"  N samples:         {len(df)}\n")
            f.write(f"  Cosine Similarity: {df['cosine_similarity'].mean():.4f} "
                    f"± {df['cosine_similarity'].std():.4f}\n")
            f.write(f"  Correlation:       {df['correlation'].mean():.4f} "
                    f"± {df['correlation'].std():.4f}\n")
            f.write(f"  Top-10 Overlap:    {df['top10_overlap'].mean():.4f} "
                    f"± {df['top10_overlap'].std():.4f}\n\n")

    # Append PKD-Skip for direct comparison
    f.write("\n" + "="*60 + "\n")
    f.write("PKD-Skip (best per size) — for comparison\n")
    f.write("="*60 + "\n\n")

    pkd_skip_models = [m for m in MODELS if m['strategy'] == 'pkd_skip']
    for model_cfg in pkd_skip_models:
        name = model_cfg["name"]
        layers = model_cfg["layers"]
        for group in ['correct', 'incorrect']:
            quant_path = Path(OUTPUT_BASE) / name / f"quantitative_{group}.csv"
            if not quant_path.exists():
                continue
            df = pd.read_csv(quant_path)
            f.write(f"{name} ({layers} layers) — [{group}]\n")
            f.write("-" * 40 + "\n")
            f.write(f"  N samples:         {len(df)}\n")
            f.write(f"  Cosine Similarity: {df['cosine_similarity'].mean():.4f} "
                    f"± {df['cosine_similarity'].std():.4f}\n")
            f.write(f"  Correlation:       {df['correlation'].mean():.4f} "
                    f"± {df['correlation'].std():.4f}\n")
            f.write(f"  Top-10 Overlap:    {df['top10_overlap'].mean():.4f} "
                    f"± {df['top10_overlap'].std():.4f}\n\n")

print(f"PKD-Last similarity report saved to: {report_path}")

PKD-Last similarity report saved to: results\interpretability_v2\pkd_last_similarity_summary.txt


In [ ]:
# ============================================================
# GENERATE README.txt FOR RESULTS FOLDER
# ============================================================
readme_path = Path(OUTPUT_BASE) / "README.txt"

readme_content = """\
Interpretability Dev v2 — Results Guide
========================================
Generated by: Interpretability_Dev_v2.ipynb
All analysis uses the held-out test set (data/casehold/test.csv).


MODELS ANALYZED
---------------
- teacher            (12 layers) — Legal-BERT fine-tuned baseline
- vanilla_L6_A0p7_T20 (6 layers) — Vanilla Knowledge Distillation, alpha=0.7, T=20
- pkd_skip_L6_B1000  (6 layers)  — PKD Skip strategy, beta=1000
- pkd_skip_L4_B100   (4 layers)  — PKD Skip strategy, beta=100
- pkd_skip_L3_B1000  (3 layers)  — PKD Skip strategy, beta=1000
- pkd_skip_L2_B500   (2 layers)  — PKD Skip strategy, beta=500
- pkd_last_L6_B500   (6 layers)  — PKD Last strategy, beta=500
- pkd_last_L4_B500   (4 layers)  — PKD Last strategy, beta=500
- pkd_last_L3_B500   (3 layers)  — PKD Last strategy, beta=500
- pkd_last_L2_B500   (2 layers)  — PKD Last strategy, beta=500

PKD-Skip layers are matched to evenly-spaced teacher layers (e.g., layers 0,2,4,6,8,10
for a 6-layer student). PKD-Last layers are matched to the final k teacher layers.


----------------------------------------------------------------------
OUTCOME VARIABLES — PREDICTIONS  ({model_name}/predictions.csv)
----------------------------------------------------------------------

example_idx
  Index of the test example (0-based, aligned to test.csv row order).

true_label
  Ground-truth answer index (0–4) from test.csv.

predicted_label
  The model's argmax prediction over 5 candidate holdings.

is_correct
  Boolean: true_label == predicted_label. Primary accuracy signal.

confidence
  Max softmax probability across the 5 choices. Reflects how certain the
  model is about its chosen answer. High confidence on incorrect predictions
  indicates overconfidence ("confident mistakes"). Compare correct vs. incorrect
  groups to assess calibration.

inference_time_sec
  Wall-clock seconds per example, averaged over the full prediction pass.
  Use to compare computational cost across model sizes.

prob_0 .. prob_4
  Full softmax probability distribution over all 5 candidate holdings.
  Used to compute KL divergence between teacher and student distributions.
  A student that picks the right answer but with a very different distribution
  from the teacher is less faithfully distilled than one whose uncertainty
  pattern also matches.


----------------------------------------------------------------------
OUTCOME VARIABLES — ATTRIBUTION QUANTITATIVE
  ({model_name}/quantitative_correct.csv, quantitative_incorrect.csv)
----------------------------------------------------------------------

For CORRECT predictions: both teacher and student run Integrated Gradients
  on the true-label input sequence.
For INCORRECT predictions: teacher runs on the true-label input; student
  runs on its predicted (wrong) label input. This makes the comparison more
  meaningful — teacher explains why the right answer is right; student
  explains why the WRONG answer seemed right to it.

cosine_similarity
  Cosine similarity between teacher and student attribution vectors. Range
  [-1, 1]; higher = more similar reasoning patterns. Correct examples tend
  to be higher because teacher and student agree on the answer. Incorrect
  examples are lower and more diagnostically useful.

correlation
  Pearson correlation between teacher and student attribution vectors.
  Similar interpretation to cosine_similarity but mean-centered. Values near
  1.0 mean the models rank token importance in the same relative order.

top10_overlap
  Fraction of the top-10 most-salient tokens (by absolute attribution) shared
  between teacher and student. Range [0, 1]. A practical measure of whether
  the models "look at" the same words. Less sensitive to magnitude differences
  than cosine_similarity.

teacher_entropy
  Shannon entropy of the teacher's absolute attribution distribution:
    H = -sum( |a_i|/sum(|a|) * log(|a_i|/sum(|a|)) )
  High entropy = teacher spreads attention broadly across many tokens.
  Low entropy  = teacher concentrates attention on a few key tokens.
  Lower entropy generally reflects more decisive, focused reasoning.

student_entropy
  Same metric for the student model. A student with much higher entropy than
  the teacher relies on more diffuse, less focused features — suggesting weaker
  alignment with the teacher's reasoning strategy. The gap
  (student_entropy - teacher_entropy) quantifies how much compression has
  degraded attentional focus.


----------------------------------------------------------------------
OUTCOME VARIABLES — SUMMARY  (summary_all_models.csv)
----------------------------------------------------------------------

model_name / strategy / layers
  Model identifier, distillation method, and number of encoder layers.

error_group
  "correct" or "incorrect" — which subset of predictions the row describes.

n_samples
  Number of test examples in this error group for this model.

mean_cosine_sim / mean_correlation / mean_top10_overlap
  Averages of the quantitative attribution metrics over the sampled examples
  in this error group.

mean_teacher_entropy / mean_student_entropy
  Average attribution entropy for teacher and student in this group.
  The gap (mean_student_entropy - mean_teacher_entropy) indicates how much
  more diffuse the student's attention is compared to the teacher.

mean_kl_divergence
  Mean KL divergence KL(teacher || student) over all test examples, using
  full softmax distributions (prob_0..prob_4). Measures how different the
  student's output distribution is from the teacher's, regardless of whether
  the final answer is correct. Higher KL = less faithfully distilled. This is
  computed at the model level (same value appears in both error_group rows).

mean_confidence
  Average max softmax probability for examples in this error group. Compare
  correct vs. incorrect rows: well-calibrated models should have substantially
  higher confidence on correct predictions.

mean_inference_time_sec
  Average per-example wall-clock time. Constant within a model. Compare across
  layer counts to quantify the speed-accuracy trade-off.

overall_accuracy
  Fraction of all test examples correctly predicted. Model-level metric (same
  value in both error_group rows for a given model).


----------------------------------------------------------------------
QUALITATIVE FILES  ({model_name}/qualitative/{correct,incorrect}/sample_NNN/)
----------------------------------------------------------------------

attribution_heatmap_filtered.png
  Side-by-side teacher (top) and student (bottom) attribution heatmaps for the
  chosen input sequence. Color intensity shows how much each token contributed
  to the model's prediction. Red = positive attribution (pushes toward predicted
  class); blue = negative. Use to visually inspect whether the student attends
  to the same legal keywords as the teacher. For incorrect examples, the student
  panel reflects its wrong-choice input; token labels come from the teacher's
  (correct) choice, so the shared context portion is directly comparable.

top15_tokens_combined.png
  Bar chart of the union of top-15 most important tokens for teacher and student.
  Shows attribution magnitudes side by side. Useful for spotting tokens the
  teacher considers important that the student ignores, or spurious tokens the
  student over-weights.

metadata.json
  Per-sample context: the case text, the 5 candidate holdings, the true label,
  predicted label, is_correct flag, confidence, inference time, and which choice
  each model attributed (teacher_attribution_choice always = true_label;
  student_attribution_choice = true_label if correct, else predicted_label).


----------------------------------------------------------------------
SUMMARY REPORTS
----------------------------------------------------------------------

pkd_last_similarity_summary.txt
  Attribution similarity statistics (cosine, correlation, top10 overlap) for
  all PKD-Last models, split by error group, followed by PKD-Skip results for
  direct comparison. Use as a write-up-ready table.

plots/accuracy_by_model.png
  Bar chart of overall test accuracy by model, colored by strategy.

plots/confidence_correct_vs_incorrect.png
  Paired bars showing mean confidence on correct vs. incorrect predictions per
  model. A large gap indicates good calibration; a small gap indicates
  overconfidence on wrong answers.

plots/cosine_sim_by_layers.png
  Line plot of mean cosine attribution similarity vs. layer count, with separate
  lines for PKD-Skip and PKD-Last. Shows whether fewer layers leads to less
  teacher-aligned reasoning, and whether the two PKD strategies diverge at
  different compression levels.

plots/inference_time_by_model.png
  Bar chart of per-example inference time ordered by layer count. Directly shows
  the latency reduction achieved by each compression level.
"""

with open(readme_path, 'w', encoding='utf-8') as f:
    f.write(readme_content)

print(f"README.txt written to: {readme_path}")